# PostgreSQL GeoDataFrame Processing Tool
## For: *list table name being cleaned here*

This notebook provides functionality to:
1. Connect to a PostgreSQL database
2. Search for tables
3. Pull tables and convert to GeoDataFrames
4. Explore and clean data
5. Push cleaned data back to PostgreSQL

In [ ]:
import os
import string
import json
import psycopg2
from psycopg2 import sql

# Database connection
conn = psycopg2.connect(
    dbname=os.environ["PGDATABASE"],
    user=os.environ["PGUSER"],
    password=os.environ["PGPASSWORD"],
    host=os.environ["PGHOST"],
    port="5432"
)

In [ ]:
cur = conn.cursor()
query = sql.SQL("""
            SELECT json_build_object(
                'type', 'FeatureCollection',
                'features', json_agg(
                    json_build_object(
                        'type', 'Feature',
                        'geometry', ST_AsGeoJSON(ST_Transform(geometry, 4326))::json,
                        'properties', to_jsonb(t) - 'geometry'
                    )
                )
            )
    FROM {} t;
""").format(sql.Identifier('model_final_results'))

cur.execute(query)

geojson = cur.fetchone()[0] 

In [ ]:
geojson

In [ ]:
# Setup and import libraries
import pandas as pd
import geopandas as gpd
import psycopg2
from psycopg2 import extras
from shapely import wkt
import shapely
from sqlalchemy import create_engine
from sqlalchemy import text

# Database connection + parameters

In [ ]:
import os
# Database connection parameters
DB_NAME = os.environ["PGDATABASE"]
DB_USER = os.environ["PGUSER"]
DB_PASSWORD = os.environ["PGPASSWORD"]
DB_HOST = os.environ["PGHOST"]
DB_PORT = os.environ["PGPORT"]

# Connect to PostgreSQL
connection = psycopg2.connect(
    database = DB_NAME,
    user = DB_USER,
    password = DB_PASSWORD,
    host = DB_HOST,
    port = DB_PORT
)
cursor = connection.cursor()

# SQLAlchemy connection string
connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# Search for tables

In [ ]:
def list_tables(search_term=None):
    """List tables in the PostgreSQL database"""
    cursor.execute("""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'public';
    """)
    
    tables = cursor.fetchall()
    
    print("Tables in the database:")
    for table in tables:
        print(table[0])

list_tables()

# Pull table of choice and convert to gdf

In [ ]:
def get_table_as_gdf(table_name, geom_col='geometry', crs=26915):
    """Pull a table from PostgreSQL and convert to GeoDataFrame"""
    # Clear any previous errors
    connection.rollback()
    
    # Check if geometry column exists
    cursor.execute(f"""
        SELECT column_name 
        FROM information_schema.columns 
        WHERE table_schema = 'public' AND table_name = '{table_name}';
    """)
    columns = [col[0] for col in cursor.fetchall()]
    
    if geom_col in columns:
        # Get data with geometry in WKT format
        cursor.execute(f"SELECT *, ST_AsText({geom_col}) as geometry_wkt FROM {table_name};")
    else:
        # Get data without geometry
        cursor.execute(f"SELECT * FROM {table_name};")
    
    rows = cursor.fetchall()
    
    # Get column names
    colnames = [desc[0] for desc in cursor.description]
    
    # Convert to DataFrame
    df = pd.DataFrame(rows, columns=colnames)
    
    # Convert to GeoDataFrame if geometry exists
    if 'geometry_wkt' in df.columns:
        # Convert WKT to shapely geometries
        df["geometry"] = df["geometry_wkt"].apply(wkt.loads)
        gdf = gpd.GeoDataFrame(df, geometry="geometry")
        gdf.set_crs(epsg=crs, inplace=True)
        return gdf
    else:
        print(f"No geometry column found in table '{table_name}'")
        return df

In [ ]:
# Choose table and convert to gdf
TABLE_NAME = "*insert table name here*"  # Change this to your table of choice
gdf = get_table_as_gdf(TABLE_NAME)
gdf

# Assess and clean data

In [ ]:
# View column names
print("Available columns:")
print(gdf.columns)

In [ ]:
# View subset of columns
COLUMNS_TO_VIEW = ['list','columns','here']  # Change this to columns you want
columns_present = [col for col in COLUMNS_TO_VIEW if col in gdf.columns]
gdf[columns_present]

In [ ]:
# View unique values in a column
COLUMN_TO_CHECK = "*insert column name here*"  # Change this to any column you want to check
if COLUMN_TO_CHECK in gdf.columns:
    print(f"Unique values in {COLUMN_TO_CHECK}:")
    print(gdf[COLUMN_TO_CHECK].unique())

In [ ]:
# Clean data: Select and rename columns

# Columns to keep (change as needed).
# Dont keep geometry.
# Dont need to list geometry_wkt because it will be automatically kept with the appned.  
COLUMNS_TO_KEEP = [
    'list','columns','to','keep','here'
]

# Filter to only existing columns
existing_columns = [col for col in COLUMNS_TO_KEEP if col in gdf.columns]
existing_columns.append("geometry_wkt")  # Always keep geometry_wkt

# Select columns
clean_gdf = gdf[existing_columns].copy()

# Rename columns (if needed)
COLUMN_RENAMES = {
    # "old_name": "new_name",
    # Example: "zip": "postal_code"
}
clean_gdf = clean_gdf.rename(columns=COLUMN_RENAMES)

# Convert all column names to lowercase
clean_gdf.columns = [col.lower() for col in clean_gdf.columns]

# View cleaned df
clean_gdf

# Push cleaned data back to postgreSQL db

In [ ]:
def push_gdf_to_db(gdf, table_name, if_exists='replace', crs=26915):
    """Push GeoDataFrame to PostgreSQL database"""
    # Make a copy to avoid modifying the original
    df_for_sql = gdf.copy()
    
    # Remove geometry column if it exists
    if 'geometry' in df_for_sql.columns:
        df_for_sql = df_for_sql.drop(columns=['geometry'])
    
    # Push to PostgreSQL
    df_for_sql.to_sql(table_name, engine, if_exists=if_exists, index=False)
    
    # Create SQL to convert WKT to PostGIS geometry
    if 'geometry_wkt' in df_for_sql.columns:
        create_geom_query = f"""
        ALTER TABLE {table_name} ADD COLUMN IF NOT EXISTS geometry geometry;
        UPDATE {table_name} SET geometry = ST_GeomFromText(geometry_wkt, {crs});
        """
        
        with engine.connect() as conn:
            conn.execute(text(create_geom_query))
            conn.commit()
        
        print(f"Table '{table_name}' created/updated with geometry column.")
    else:
        print(f"Table '{table_name}' created/updated without geometry column.")

In [ ]:
# Set output table name and push data
OUTPUT_TABLE = f"clean_{TABLE_NAME}"  # Automatically add "clean_" prefix to original table name
push_gdf_to_db(clean_gdf, OUTPUT_TABLE)

In [ ]:
# Verify the output table was created
cursor.execute("""
    SELECT EXISTS (
        SELECT 1 FROM information_schema.tables 
        WHERE table_schema = 'public' AND table_name = %s
    );
""", (OUTPUT_TABLE,))

table_exists = cursor.fetchone()[0]

list_tables()

if table_exists:
    print(f"\nSuccess! The table '{OUTPUT_TABLE}' has been created and is in the database.")
else:
    print(f"\nWarning: The table '{OUTPUT_TABLE}' was not found in the database. Check for errors.")

In [ ]:
# Close connection
cursor.close()
connection.close()
print("Database connection closed.")